# 04 — Random Forest

Second model. Random Forest captures non-linear interactions between features
(e.g. beach station + hot day + Sunday) that Linear Regression misses.

**Cross-validation:** `TimeSeriesSplit` with a gap to avoid leakage between
train and validation folds.

**Hyperparameter search:** `RandomizedSearchCV` on a 500k-row sample of the
training set. Full ~16M train rows aren't needed to find good hyperparameters —
500k gives stable CV estimates in a fraction of the time. The winning
hyperparameters are then used to train the final model on the full train set.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint

from src.utils.config import PATHS, FEATURES, TARGET, SPLIT_DATE, RANDOM_STATE, DATASET_FILE
from src.models.evaluate import (
    regression_metrics, plot_predictions, plot_residuals, plot_feature_importance
)

print('Libraries loaded OK')

---
## 1. Load and split

In [ ]:
df = pd.read_parquet(DATASET_FILE)
df['datetime_hour'] = pd.to_datetime(df['datetime_hour'])

train = df[df['datetime_hour'] < SPLIT_DATE]
test  = df[df['datetime_hour'] >= SPLIT_DATE]

X_test, y_test = test[FEATURES], test[TARGET]
avg_capacity = float(df['capacity'].drop_duplicates().mean())

print(f'Full train: {len(train):,}  Test: {len(test):,}')

---
## 2. Sample for hyperparameter search

Using the full ~16M train rows for CV would take hours per fit.
A stratified 500k sample covers all horizons proportionally and
gives stable enough CV estimates to find good hyperparameters.

In [ ]:
SEARCH_SAMPLE = 500_000

# Sample proportionally within each horizon so all are represented equally
train_sample = (
    train
    .groupby('horizon_hours', group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), SEARCH_SAMPLE // train['horizon_hours'].nunique()),
        random_state=RANDOM_STATE
    ))
)

X_search = train_sample[FEATURES]
y_search = train_sample[TARGET]

# Full train arrays used later to fit the final model with best params
X_train = train[FEATURES]
y_train = train[TARGET]

print(f'Search sample: {len(train_sample):,} rows ({len(train_sample)/len(train)*100:.1f}% of train)')

---
## 3. Hyperparameter search

In [ ]:
# TimeSeriesSplit ensures folds always respect time order.
# gap=168 (one week) keeps the end of each train fold and the start
# of the next val fold separated, reducing autocorrelation bleed-through.
tscv = TimeSeriesSplit(n_splits=3, gap=168)

param_dist = {
    # More trees = more stable predictions, but returns diminish past ~300.
    'n_estimators':      [100, 200, 300],

    # None lets trees grow until leaves are pure — often overfits on large datasets.
    # Capping at 8-16 acts as regularization and keeps inference fast.
    'max_depth':         [8, 12, 16],

    # Min samples required to split a node. Higher values prevent the tree from
    # learning very specific patterns that don't generalise.
    'min_samples_split': [10, 20, 40],

    # Min samples required in a leaf. Similar effect to min_samples_split but
    # applied at the bottom of the tree — smooths out noisy leaves.
    'min_samples_leaf':  [5, 10, 20],

    # Fraction of features considered at each split. sqrt is the classic default;
    # lower values add more randomness between trees, which reduces correlation
    # between them and often improves ensemble performance.
    'max_features':      ['sqrt', 0.5],
}

# n_jobs=-1 on the RF so it parallelises across trees.
# n_jobs=1 on the search because on Windows, having both the outer search
# and the inner RF try to spawn workers causes a PicklingError.
rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    # 10 iterations × 3 folds = 30 fits on 500k rows — runs in ~30-60 min.
    n_iter=10,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE,
    verbose=2,
    n_jobs=1,
)

search.fit(X_search, y_search)
print(f'Best CV RMSE: {-search.best_score_:.4f}')
print(f'Best params:  {search.best_params_}')

---
## 4. Retrain on full train set with best hyperparameters

The search found the best config on a sample. Now we train the final
model on all available training data to get the best possible performance.

In [ ]:
best_rf = RandomForestRegressor(
    **search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)
print('Final model trained on full train set.')

---
## 5. Evaluate

In [ ]:
y_pred_train = best_rf.predict(X_train)
y_pred_test  = best_rf.predict(X_test)

print('=== RANDOM FOREST ===')
m_train = regression_metrics(y_train, y_pred_train, 'Train', avg_capacity)
print()
m_test  = regression_metrics(y_test, y_pred_test, 'Test', avg_capacity)

---
## 6. RMSE by horizon

In [ ]:
from sklearn.metrics import mean_squared_error

rows = []
for h in sorted(test['horizon_hours'].unique()):
    mask = test['horizon_hours'] == h
    rmse = float(np.sqrt(mean_squared_error(y_test[mask], y_pred_test[mask])))
    rows.append({'horizon_hours': h, 'rmse': rmse})

df_h = pd.DataFrame(rows)
print(df_h.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_h['horizon_hours'], df_h['rmse'], marker='o', color='steelblue')
ax.set_xlabel('Horizon (hours ahead)')
ax.set_ylabel('RMSE')
ax.set_title('Random Forest — RMSE by prediction horizon')
plt.tight_layout()
plt.savefig(PATHS['figures'] / '04_rf_rmse_by_horizon.png', dpi=150)
plt.show()

---
## 7. Feature importances

In [ ]:
fig = plot_feature_importance(FEATURES, best_rf.feature_importances_,
                              title='Random Forest — feature importances')
fig.savefig(PATHS['figures'] / '04_rf_feature_importance.png', dpi=150)
plt.show()

---
## 8. Prediction plots

In [ ]:
fig = plot_predictions(y_test.values, y_pred_test,
                       rmse=m_test['rmse'], r2=m_test['r2'],
                       title='Random Forest — Predicted vs Actual')
fig.savefig(PATHS['figures'] / '04_rf_predictions.png', dpi=150)
plt.show()

fig = plot_residuals(y_test.values, y_pred_test, title='Random Forest — Residuals')
fig.savefig(PATHS['figures'] / '04_rf_residuals.png', dpi=150)
plt.show()

---
## 9. Save model

In [ ]:
model_path = PATHS['models'] / 'random_forest.joblib'
joblib.dump(best_rf, model_path)
print(f'Model saved to {model_path}')